# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll enumerate the record sets and their fields, noting all `@id` values as required for future referencing.

In [ ]:
# List all record sets
record_sets = []
for rs in metadata.record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id        : {rs.id}")
    print(f"  Description: {rs.description}")
    fields = rs.fields
    print(f"  Fields:")
    for field in fields:
        print(f"    - Name: {field.name}")
        print(f"      @id : {field.id}")
        print(f"      DataType: {field.data_type}")
    print()
    record_sets.append(rs.id)

# Display available record set IDs
print("Available record set @id values:")
for r in record_sets:
    print(f" - {r}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set by its `@id`

# You can edit this list if you wish to select only a subset
record_set_ids = record_sets
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set {record_set_id}")
        print(f"Fields: {df.columns.tolist()}")
        display(df.head(3))
    else:
        print(f"No records found for record set {record_set_id}")

# For demonstration, select the first record set for further analysis
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    df = dataframes.get(main_record_set_id, None)
    if df is not None:
        print(f"Columns for record set {main_record_set_id}:")
        print(df.columns.tolist())
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates filtering, normalization, and grouping for key numeric fields by referencing the appropriate `@id`.

In [ ]:
# For this example, we'll use common proteomics fields such as 'MW [kDa]', 'Coverage [%]', 'Unique Peptides'.
# These fields should be referenced by their @id, as printed in the overview above.
# Please adjust these to match the actual field @ids in your dataset.

import numpy as np

if df is not None and len(df.columns) > 0:
    # Try to automatically detect a numeric field by dtype if possible
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric field candidates: {numeric_candidates}")
    if len(numeric_candidates) > 0:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}")
    else:
        # Fallback: try some well-known fields by name
        for candidate in ['MW [kDa]', 'Coverage [%]', 'Unique Peptides', 'Peptides', 'Molec. Weight', 'molecular_weight', 'id']:
            if candidate in df.columns:
                numeric_field = candidate
                print(f"Using fallback numeric field: {numeric_field}")
                break
        else:
            print("No numeric field found. EDA cannot continue.")
            numeric_field = None

    if numeric_field is not None:
        # Example filter: take values above median
        threshold = df[numeric_field].median()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize the field (z-score)
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / 
            filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a likely categorical field
        group_field = None
        for candidate in ['Protein Group', 'Description', 'Sample', 'condition', 'category']:
            if candidate in df.columns:
                group_field = candidate
                print(f"Grouping by: {group_field}")
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouping field is available, boxplot
    if group_field:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded, explored, and visually analyzed the FAIR<sup>2</sup> mass spectrometry dataset using the `mlcroissant` library. We demonstrated data loading, field identification by `@id`, basic filtering, normalization, grouping, and data visualization. This workflow provides a reproducible foundation for future domain-specific analyses or integration with machine learning pipelines.
